# 📊 Customer Churn Prediction
### Telco Customer Dataset — End-to-End ML Pipeline

**Goal:** Predict whether a customer will churn (leave the service) based on usage patterns and account info.

**Models:** Logistic Regression · Random Forest · XGBoost

---

## 1. Import Libraries
`pandas` / `numpy` for data manipulation, `matplotlib` / `seaborn` for visualisation, `scikit-learn` for ML, and `xgboost` for gradient boosting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from xgboost import XGBClassifier
import joblib

os.makedirs('../data/images', exist_ok=True)
os.makedirs('../model', exist_ok=True)

print('All libraries imported successfully!')


## 2. Load the Dataset
IBM Telco Customer Churn dataset (~7,000 rows). Adjust `FILE_PATH` if your CSV name differs.

In [ ]:
FILE_PATH = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(FILE_PATH)
print(f'Shape: {df.shape}')
df.head()


## 3. Exploratory Data Analysis (EDA)
We inspect class balance, distributions of key features, and identify any data quality issues before building models.

In [ ]:
print('=== Data Types & Nulls ===')
print(df.info())
print('\n=== Churn Distribution ===')
print(df['Churn'].value_counts())
print('\n=== Descriptive Statistics (numeric) ===')
print(df.describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Pie chart
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=['No Churn','Churn'],
            autopct='%1.1f%%', colors=['#4CAF50','#F44336'], startangle=90)
axes[0].set_title('Churn Distribution')

# Tenure histogram
df[df['Churn']=='No']['tenure'].hist(ax=axes[1], alpha=0.7, label='No Churn', color='#4CAF50', bins=30)
df[df['Churn']=='Yes']['tenure'].hist(ax=axes[1], alpha=0.7, label='Churn', color='#F44336', bins=30)
axes[1].set_title('Tenure Distribution by Churn')
axes[1].set_xlabel('Tenure (months)')
axes[1].legend()

# Monthly charges boxplot
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[2])
axes[2].set_title('Monthly Charges by Churn')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../data/images/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved.')


## 4. Data Preprocessing
Steps:
1. Convert `TotalCharges` from string → numeric (spaces for new customers become NaN, filled with median)
2. Drop `customerID` (identifier, not a feature)
3. Label-encode all categorical columns
4. Scale numerical features with `StandardScaler` (required for Logistic Regression)

In [ ]:
# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop ID column
df.drop(columns=['customerID'], inplace=True)

# Encode categoricals
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Encoding columns: {cat_cols}')
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('Preprocessing complete!')
df.head()


In [ ]:
plt.figure(figsize=(16, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved.')


## 5. Train-Test Split & Scaling
80/20 stratified split ensures class proportions are preserved. `StandardScaler` is fit **only on training data** to prevent data leakage.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples | Features: {X_train.shape[1]}')
print(f'Churn rate in train: {y_train.mean():.2%}')


## 6. Model 1 — Logistic Regression (Baseline)
Logistic Regression applies a sigmoid function to a linear combination of features to output churn probability. Fast, interpretable, and a solid baseline.

In [ ]:
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_pred       = lr_model.predict(X_test_scaled)
lr_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_acc = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_pred_proba)

print(f'Accuracy : {lr_acc:.4f} | ROC-AUC : {lr_auc:.4f}')
print(classification_report(y_test, lr_pred, target_names=['No Churn','Churn']))


## 7. Model 2 — Random Forest
An ensemble of 200 decision trees. Each tree sees a random subset of the training data and features (bagging). Final prediction is by majority vote. Robust, handles non-linearity, and gives feature importance scores.

In [ ]:
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # Tree-based: no scaling needed

rf_pred       = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_pred_proba)

print(f'Accuracy : {rf_acc:.4f} | ROC-AUC : {rf_auc:.4f}')
print(classification_report(y_test, rf_pred, target_names=['No Churn','Churn']))


In [ ]:
fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=True).tail(15)

plt.figure(figsize=(10, 7))
plt.barh(fi_df['Feature'], fi_df['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Top 15 Feature Importances — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Model 3 — XGBoost
XGBoost trains trees sequentially: each new tree minimises the residual errors of previous trees (gradient boosting). Highly competitive on structured/tabular data with careful hyperparameter tuning.

In [ ]:
print('Training XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=5,
    use_label_encoder=False, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_pred       = xgb_model.predict(X_test)
xgb_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_pred_proba)

print(f'Accuracy : {xgb_acc:.4f} | ROC-AUC : {xgb_auc:.4f}')
print(classification_report(y_test, xgb_pred, target_names=['No Churn','Churn']))


## 9. Model Comparison
Side-by-side comparison of all three models using accuracy, ROC-AUC, confusion matrices, and ROC curves.

In [ ]:
results = pd.DataFrame({
    'Model':    ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [lr_acc, rf_acc, xgb_acc],
    'ROC-AUC':  [lr_auc, rf_auc, xgb_auc]
})
results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))


In [ ]:
models   = ['Logistic Regression', 'Random Forest', 'XGBoost']
accuracy = [lr_acc, rf_acc, xgb_acc]
auc      = [lr_auc, rf_auc, xgb_auc]
x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, accuracy, width, label='Accuracy',  color='steelblue',  alpha=0.85)
b2 = ax.bar(x + width/2, auc,      width, label='ROC-AUC',   color='darkorange', alpha=0.85)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy vs ROC-AUC', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('../data/images/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for name, proba in [('Logistic Regression', lr_pred_proba),
                    ('Random Forest', rf_pred_proba),
                    ('XGBoost', xgb_pred_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc_val:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold', fontsize=13)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/images/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
for ax, (name, pred) in zip(axes, [('Logistic Regression', lr_pred),
                                     ('Random Forest', rf_pred),
                                     ('XGBoost', xgb_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Churn','Churn'],
                yticklabels=['No Churn','Churn'], ax=ax)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../data/images/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Save Best Model & Scaler
The best model (by ROC-AUC), the scaler, and feature names are saved to `../model/`. These files will be loaded directly by the Flask web application.

In [ ]:
aucs = {'lr': (lr_auc, lr_model), 'rf': (rf_auc, rf_model), 'xgb': (xgb_auc, xgb_model)}
best_key = max(aucs, key=lambda k: aucs[k][0])
best_auc, best_model = aucs[best_key]

joblib.dump(best_model,        '../model/churn_model.pkl')
joblib.dump(scaler,            '../model/scaler.pkl')
joblib.dump(list(X.columns),  '../model/feature_names.pkl')

print(f'Best model : {best_key.upper()} (AUC = {best_auc:.4f})')
print('Saved: model/churn_model.pkl')
print('Saved: model/scaler.pkl')
print('Saved: model/feature_names.pkl')
print('Notebook complete! Ready for Flask integration.')
